In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


data_dir = Path("data/features/only features")
files = list(data_dir.glob("*.csv"))
stock_data = {f.stem.replace("_features", ""): pd.read_csv(f) for f in files}



def make_direction_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors = "coerce")

    for d in [5,10,20]:
        ma_col = f"MA{d}"
        ma_next = df[ma_col].shift(-1)
        dir_col = f"MA{d}_dir"
        df[dir_col] = np.where(ma_next > df[ma_col], 1 , -1)

    df = df.iloc[:-1].reset_index(drop = True)
    return df

labeled_data = {ticker: make_direction_labels(df) for ticker, df in stock_data.items()}

In [4]:
# Now we need to split the data into training and testing sets
# But since we have 3 different targets, we need to split the data for each target
# And since this is the time series data, we need to split the data by the index, not the random split

targets_dir = ["MA5_dir", "MA10_dir", "MA20_dir"]

split_data = {}

for ticker, df in labeled_data.items():

    df = df.copy()
    df = df.sort_values("Date")


    exclude_cols = ["Date"] + targets_dir

    split_idx = int(len(df) * 0.8)

    X_all = df.drop(columns=exclude_cols, errors="ignore")  
    X_train_raw = X_all.iloc[:split_idx].copy()
    X_test_raw  = X_all.iloc[split_idx:].copy()

    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), 
                           columns=X_train_raw.columns, index=X_train_raw.index)
    X_test  = pd.DataFrame(scaler.transform(X_test_raw), 
                           columns=X_test_raw.columns, index=X_test_raw.index)

    split_data[ticker] = {"X_train": X_train, "X_test": X_test}
    for t in targets_dir:
        y_all = df[t]
        y_train = y_all.iloc[:split_idx]
        y_test  = y_all.iloc[split_idx:]
        split_data[ticker][t] = {"y_train": y_train, "y_test": y_test}

In [5]:
# Running the Decision Tree Regression
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results= []
models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    models[ticker] = {}

    for target in targets_dir:
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        reg = DecisionTreeRegressor(
            random_state=42,
            max_depth=None,
            min_samples_leaf=5,
            criterion="squared_error" 
        )

        reg.fit(X_train, y_train)

        y_pred_cont = reg.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)  

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)


        results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  
            "Model": "DecisionTreeClassifier",
            "MAE": mae,                    
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        models[ticker][target] = reg

res_df = pd.DataFrame(results).sort_values(["Ticker", "Target"])

res_df["Target"] = pd.Categorical(res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
res_df = res_df.sort_values(["Ticker","Target"])

table = (
    res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)

print(table.to_string())

                                           MAE       MSE      RMSE        R2
Ticker Target Model                                                         
AAPL   MA5    DecisionTreeClassifier  0.535211  1.070423  1.034612 -0.079470
       MA10   DecisionTreeClassifier  0.366197  0.732394  0.855800  0.258188
       MA20   DecisionTreeClassifier  0.295775  0.591549  0.769122  0.387931
ADBE   MA5    DecisionTreeClassifier  0.443662  0.887324  0.941979  0.112280
       MA10   DecisionTreeClassifier  0.345070  0.690141  0.830747  0.298589
       MA20   DecisionTreeClassifier  0.232394  0.464789  0.681754  0.499359
AMD    MA5    DecisionTreeClassifier  0.725352  1.450704  1.204452 -0.453299
       MA10   DecisionTreeClassifier  0.387324  0.774648  0.880141  0.224390
       MA20   DecisionTreeClassifier  0.140845  0.281690  0.530745  0.716906
CRM    MA5    DecisionTreeClassifier  0.436620  0.873239  0.934473  0.126067
       MA10   DecisionTreeClassifier  0.338028  0.676056  0.822226  0.320574

In [6]:
# Running the SVM
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


svm_results = []
svm_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    svm_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        svm = SVR(kernel="linear", C=1.0, epsilon=0.1)
        svm.fit(X_train, y_train)

        y_pred_cont = svm.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        svm_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""), 
            "Model": "SVM",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        svm_models[ticker][target] = svm

svm_res_df = pd.DataFrame(svm_results)
svm_res_df["Target"] = pd.Categorical(svm_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
svm_table = (
    svm_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(svm_table.to_string())

                          MAE       MSE      RMSE        R2
Ticker Target Model                                        
AAPL   MA5    SVM    0.345070  0.690141  0.830747  0.304026
       MA10   SVM    0.281690  0.563380  0.750587  0.429375
       MA20   SVM    0.211268  0.422535  0.650027  0.562808
ADBE   MA5    SVM    0.288732  0.577465  0.759911  0.422277
       MA10   SVM    0.232394  0.464789  0.681754  0.527621
       MA20   SVM    0.330986  0.661972  0.813617  0.286966
AMD    MA5    SVM    0.394366  0.788732  0.888106  0.209857
       MA10   SVM    0.267606  0.535211  0.731581  0.464124
       MA20   SVM    0.176056  0.352113  0.593391  0.646132
CRM    MA5    SVM    0.316901  0.633803  0.796117  0.365694
       MA10   SVM    0.218310  0.436620  0.660772  0.561204
       MA20   SVM    0.302817  0.605634  0.778225  0.394336
MSFT   MA5    SVM    0.415493  0.830986  0.911584  0.144578
       MA10   SVM    0.253521  0.507042  0.712069  0.486438
       MA20   SVM    0.211268  0.422535 

In [7]:
# Runnding Bagging 

from sklearn.ensemble import BaggingRegressor

bag_results = []
bag_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    bag_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        base = DecisionTreeRegressor(random_state=42, min_samples_leaf=5)
        model = BaggingRegressor(
            estimator=base,        # scikit-learn >= 1.2
            n_estimators=300,
            bootstrap=True,
            n_jobs=-1,
            random_state=42
        )
        model.fit(X_train, y_train)

        y_pred_cont = model.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        bag_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  # "MA5","MA10","MA20"
            "Model": "BaggingRegressor",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        bag_models[ticker][target] = model

bag_res_df = pd.DataFrame(bag_results)
bag_res_df["Target"] = pd.Categorical(bag_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
bag_table = (
    bag_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(bag_table.to_string())

                                     MAE       MSE      RMSE        R2
Ticker Target Model                                                   
AAPL   MA5    BaggingRegressor  0.359155  0.718310  0.847532  0.275619
       MA10   BaggingRegressor  0.274648  0.549296  0.741145  0.443641
       MA20   BaggingRegressor  0.211268  0.422535  0.650027  0.562808
ADBE   MA5    BaggingRegressor  0.302817  0.605634  0.778225  0.394096
       MA10   BaggingRegressor  0.232394  0.464789  0.681754  0.527621
       MA20   BaggingRegressor  0.161972  0.323944  0.569160  0.651068
AMD    MA5    BaggingRegressor  0.323944  0.647887  0.804914  0.350954
       MA10   BaggingRegressor  0.253521  0.507042  0.712069  0.492328
       MA20   BaggingRegressor  0.140845  0.281690  0.530745  0.716906
CRM    MA5    BaggingRegressor  0.359155  0.718310  0.847532  0.281120
       MA10   BaggingRegressor  0.204225  0.408451  0.639101  0.589514
       MA20   BaggingRegressor  0.197183  0.394366  0.627986  0.605614
MSFT  

In [8]:
# Random forest

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf_results = []
rf_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    rf_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        model = RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
            bootstrap=True
        )
        model.fit(X_train, y_train)

        y_pred_cont = model.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        rf_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  # "MA5","MA10","MA20"
            "Model": "RandomForestRegressor",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        rf_models[ticker][target] = model

rf_res_df = pd.DataFrame(rf_results)
rf_res_df["Target"] = pd.Categorical(rf_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
rf_table = (
    rf_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(rf_table.to_string())

                                          MAE       MSE      RMSE        R2
Ticker Target Model                                                        
AAPL   MA5    RandomForestRegressor  0.359155  0.718310  0.847532  0.275619
       MA10   RandomForestRegressor  0.281690  0.563380  0.750587  0.429375
       MA20   RandomForestRegressor  0.211268  0.422535  0.650027  0.562808
ADBE   MA5    RandomForestRegressor  0.338028  0.676056  0.822226  0.323642
       MA10   RandomForestRegressor  0.218310  0.436620  0.660772  0.556250
       MA20   RandomForestRegressor  0.183099  0.366197  0.605142  0.605556
AMD    MA5    RandomForestRegressor  0.323944  0.647887  0.804914  0.350954
       MA10   RandomForestRegressor  0.274648  0.549296  0.741145  0.450022
       MA20   RandomForestRegressor  0.147887  0.295775  0.543852  0.702751
CRM    MA5    RandomForestRegressor  0.366197  0.732394  0.855800  0.267024
       MA10   RandomForestRegressor  0.204225  0.408451  0.639101  0.589514
       MA20 

In [ ]:
# Ada boost
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ada_results = []
ada_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    ada_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        weak = DecisionTreeRegressor(max_depth=3, random_state=42)

        try:
            model = AdaBoostRegressor(
                estimator=weak,           # scikit-learn >= 1.2
                n_estimators=300,
                learning_rate=0.1,
                random_state=42
            )
        except TypeError:
            model = AdaBoostRegressor(
                base_estimator=weak,      # scikit-learn < 1.2
                n_estimators=300,
                learning_rate=0.1,
                random_state=42
            )

        model.fit(X_train, y_train)

        y_pred_cont = model.predict(X_test)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        ada_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  # "MA5","MA10","MA20"
            "Model": "AdaBoostRegressor(Depth3Tree)",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        ada_models[ticker][target] = model

ada_res_df = pd.DataFrame(ada_results)
ada_res_df["Target"] = pd.Categorical(ada_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
ada_table = (
    ada_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(ada_table.to_string())

                                                  MAE       MSE      RMSE        R2
Ticker Target Model                                                                
AAPL   MA5    AdaBoostRegressor(Depth3Tree)  0.316901  0.633803  0.796117  0.360840
       MA10   AdaBoostRegressor(Depth3Tree)  0.295775  0.591549  0.769122  0.400844
       MA20   AdaBoostRegressor(Depth3Tree)  0.211268  0.422535  0.650027  0.562808
ADBE   MA5    AdaBoostRegressor(Depth3Tree)  0.288732  0.577465  0.759911  0.422277
       MA10   AdaBoostRegressor(Depth3Tree)  0.253521  0.507042  0.712069  0.484677
       MA20   AdaBoostRegressor(Depth3Tree)  0.253521  0.507042  0.712069  0.453846
AMD    MA5    AdaBoostRegressor(Depth3Tree)  0.309859  0.619718  0.787222  0.379173
       MA10   AdaBoostRegressor(Depth3Tree)  0.260563  0.521127  0.721891  0.478226
       MA20   AdaBoostRegressor(Depth3Tree)  0.119718  0.239437  0.489323  0.759370
CRM    MA5    AdaBoostRegressor(Depth3Tree)  0.309859  0.619718  0.787222  0

In [ ]:
# Cat boost
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor, Pool


cat_results = []
cat_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    cat_models[ticker] = {}

    for target in targets_dir:  # ["MA5_dir","MA10_dir","MA20_dir"]
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        train_pool = Pool(X_train, y_train)
        test_pool  = Pool(X_test,  y_test)

        model = CatBoostRegressor(
            iterations=300,
            depth=6,
            learning_rate=0.1,
            loss_function="RMSE",
            random_seed=42,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1
        )

        model.fit(train_pool)

        y_pred_cont = model.predict(test_pool)
        y_pred = np.where(y_pred_cont > 0, 1, -1)

        mae  = mean_absolute_error(y_test, y_pred)
        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        cat_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),  # "MA5","MA10","MA20"
            "Model": "CatBoostRegressor",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        cat_models[ticker][target] = model

cat_res_df = pd.DataFrame(cat_results)
cat_res_df["Target"] = pd.Categorical(cat_res_df["Target"], categories=["MA5","MA10","MA20"], ordered=True)
cat_table = (
    cat_res_df
    .set_index(["Ticker","Target","Model"])
    [["MAE","MSE","RMSE","R2"]]
    .sort_index()
)
print(cat_table.to_string())

                                      MAE       MSE      RMSE        R2
Ticker Target Model                                                    
AAPL   MA5    CatBoostRegressor  0.352113  0.704225  0.839181  0.289822
       MA10   CatBoostRegressor  0.267606  0.535211  0.731581  0.457906
       MA20   CatBoostRegressor  0.260563  0.521127  0.721891  0.460796
ADBE   MA5    CatBoostRegressor  0.323944  0.647887  0.804914  0.351823
       MA10   CatBoostRegressor  0.260563  0.521127  0.721891  0.470363
       MA20   CatBoostRegressor  0.197183  0.394366  0.627986  0.575214
AMD    MA5    CatBoostRegressor  0.338028  0.676056  0.822226  0.322734
       MA10   CatBoostRegressor  0.288732  0.577465  0.759911  0.421818
       MA20   CatBoostRegressor  0.133803  0.267606  0.517306  0.731061
CRM    MA5    CatBoostRegressor  0.401408  0.802817  0.896001  0.196546
       MA10   CatBoostRegressor  0.211268  0.422535  0.650027  0.575359
       MA20   CatBoostRegressor  0.211268  0.422535  0.650027  0

In [11]:
# xbg boost

from xgboost import XGBClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

xgb_results = []
xgb_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    xgb_models[ticker] = {}

    for target in targets_dir:
        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        y_train_bin = (y_train == 1).astype(int)
        y_test_bin  = (y_test == 1).astype(int)

        xgb = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            random_state=42
        )

        xgb.fit(X_train, y_train_bin)

        y_pred_bin = xgb.predict(X_test)

        y_pred = np.where(y_pred_bin == 1, 1, -1)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        xgb_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),
            "Model": "XGBoostClassifier",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        xgb_models[ticker][target] = xgb

xgb_df = pd.DataFrame(xgb_results)
print(xgb_df.to_string())


   Ticker Target              Model       MAE       MSE      RMSE        R2
0    AAPL    MA5  XGBoostClassifier  0.380282  0.760563  0.872103  0.233008
1    AAPL   MA10  XGBoostClassifier  0.274648  0.549296  0.741145  0.443641
2    AAPL   MA20  XGBoostClassifier  0.246479  0.492958  0.702109  0.489943
3    ADBE    MA5  XGBoostClassifier  0.316901  0.633803  0.796117  0.365914
4    ADBE   MA10  XGBoostClassifier  0.281690  0.563380  0.750587  0.427419
5    ADBE   MA20  XGBoostClassifier  0.176056  0.352113  0.593391  0.620726
6     AMD    MA5  XGBoostClassifier  0.366197  0.732394  0.855800  0.266296
7     AMD   MA10  XGBoostClassifier  0.274648  0.549296  0.741145  0.450022
8     AMD   MA20  XGBoostClassifier  0.140845  0.281690  0.530745  0.716906
9     CRM    MA5  XGBoostClassifier  0.387324  0.774648  0.880141  0.224737
10    CRM   MA10  XGBoostClassifier  0.218310  0.436620  0.660772  0.561204
11    CRM   MA20  XGBoostClassifier  0.218310  0.436620  0.660772  0.563359
12   MSFT   

In [ ]:
# lgb boost

from lightgbm import LGBMClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lgb_results = []
lgb_models = {}

for ticker, parts in split_data.items():
    X_train = parts["X_train"]
    X_test  = parts["X_test"]

    lgb_models[ticker] = {}

    for target in targets_dir:

        y_train = parts[target]["y_train"]
        y_test  = parts[target]["y_test"]

        y_train_bin = (y_train == 1).astype(int)
        y_test_bin  = (y_test == 1).astype(int)

        lgb = LGBMClassifier(
            n_estimators=400,
            learning_rate=0.03,
            max_depth=-1,         
            num_leaves=31,        
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            verbose = -1
        )

        lgb.fit(X_train, y_train_bin)

        y_pred_bin = lgb.predict(X_test)

        y_pred = np.where(y_pred_bin == 1, 1, -1)

        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)

        lgb_results.append({
            "Ticker": ticker,
            "Target": target.replace("_dir",""),
            "Model": "LightGBMClassifier",
            "MAE": mae,
            "MSE": mse,
            "RMSE": rmse,
            "R2": r2
        })

        lgb_models[ticker][target] = lgb

lgb_df = pd.DataFrame(lgb_results)
print(lgb_df.to_string())


   Ticker Target               Model       MAE       MSE      RMSE        R2
0    AAPL    MA5  LightGBMClassifier  0.373239  0.746479  0.863990  0.247212
1    AAPL   MA10  LightGBMClassifier  0.274648  0.549296  0.741145  0.443641
2    AAPL   MA20  LightGBMClassifier  0.253521  0.507042  0.712069  0.475369
3    ADBE    MA5  LightGBMClassifier  0.323944  0.647887  0.804914  0.351823
4    ADBE   MA10  LightGBMClassifier  0.295775  0.591549  0.769122  0.398790
5    ADBE   MA20  LightGBMClassifier  0.232394  0.464789  0.681754  0.499359
6     AMD    MA5  LightGBMClassifier  0.366197  0.732394  0.855800  0.266296
7     AMD   MA10  LightGBMClassifier  0.253521  0.507042  0.712069  0.492328
8     AMD   MA20  LightGBMClassifier  0.126761  0.253521  0.503509  0.745215
9     CRM    MA5  LightGBMClassifier  0.359155  0.718310  0.847532  0.281120
10    CRM   MA10  LightGBMClassifier  0.232394  0.464789  0.681754  0.532895
11    CRM   MA20  LightGBMClassifier  0.211268  0.422535  0.650027  0.577444

In [ ]:
tables = [
    table,
    svm_table,
    bag_table,
    rf_table,
    ada_table,
    cat_table,
    xgb_df,
    lgb_df
]

clean_tables = []
for t in tables:
    df = t.reset_index()         
    df.columns = df.columns.astype(str)
    clean_tables.append(df)


combined_df = pd.concat(clean_tables, ignore_index=True)

model_avg = (
    combined_df
    .groupby("Model")[["MAE", "MSE", "RMSE", "R2"]]
    .mean()
    .sort_values("MAE")
)

print(model_avg.to_string())

                                    MAE       MSE      RMSE        R2
Model                                                                
AdaBoostRegressor(Depth3Tree)  0.254988  0.509977  0.706176  0.479664
BaggingRegressor               0.256749  0.513498  0.707886  0.476749
RandomForestRegressor          0.270833  0.541667  0.726304  0.447963
XGBoostClassifier              0.275235  0.550469  0.733594  0.439265
LightGBMClassifier             0.279343  0.558685  0.740186  0.430689
CatBoostRegressor              0.285798  0.571596  0.747294  0.417410
SVM                            0.303697  0.607394  0.772551  0.379210
DecisionTreeClassifier         0.398768  0.797535  0.879419  0.187864


In [ ]:
def backtest(prices, signals, initial_capital=100000):
    position = 0
    cash = initial_capital
    shares = 0
    
    for i in range(len(signals)):
        if signals[i] == 1 and position == 0:
            shares = cash / prices[i]
            cash = 0
            position = 1
        elif signals[i] == -1 and position == 1:
            cash = shares * prices[i]
            shares = 0
            position = 0
    
    final_value = cash + shares * prices[-1]
    return final_value


all_models = {
    "DecisionTree": models,
    "SVM": svm_models,
    "Bagging": bag_models,
    "RandomForest": rf_models,
    "AdaBoost": ada_models,
    "CatBoost": cat_models,
    "XGBoost": xgb_models,
    "LightGBM": lgb_models
}

targets = ["MA5_dir", "MA10_dir", "MA20_dir"]

results = []

for ticker in split_data.keys():

    df = labeled_data[ticker]
    test_len = len(split_data[ticker]["X_test"])
    close_prices = df["Close"].iloc[-test_len:].values

    for target in targets:
        for model_name, model_dict in all_models.items():

            model = model_dict[ticker][target]
            y_pred_cont = model.predict(split_data[ticker]["X_test"])
            signals = np.where(y_pred_cont > 0, 1, -1)

            final_value = backtest(close_prices, signals)
            total_return = (final_value - 100000) / 100000 * 100

            results.append({
                "Ticker": ticker,
                "Target": target.replace("_dir",""),  
                "Model": model_name,
                "Return%": round(total_return, 2)
            })

backtest_df = pd.DataFrame(results)


for ma in ["MA5", "MA10", "MA20"]:
    print(f"\n===================== {ma} Result =====================\n")
    
    tmp = backtest_df[backtest_df["Target"] == ma]
    
    pivot = tmp.pivot_table(
        index="Ticker",    
        columns="Model",   
        values="Return%", 
        aggfunc="mean"
    )
    
    display(pivot)



===================== MA5 Result =====================



Model,AdaBoost,Bagging,CatBoost,DecisionTree,LightGBM,RandomForest,SVM,XGBoost
Ticker,,,,,,,,
AAPL,11.86,5.32,19.44,6.08,-0.57,9.64,-0.13,4.23
ADBE,-3.37,-7.24,-13.32,-22.31,-15.10,-14.33,-6.01,-17.84
AMD,13.33,9.78,-10.36,41.41,4.02,7.83,-5.26,7.87
CRM,3.84,2.94,4.38,-2.46,1.93,0.25,11.03,-0.15
MSFT,9.63,10.06,13.19,22.06,0.77,11.89,17.53,7.19
NOW,-13.22,-6.29,-22.24,-2.37,-18.37,0.15,-12.45,-8.67
NVDA,25.67,42.68,-6.70,52.74,36.10,47.57,35.22,27.50
ORCL,69.17,49.61,77.45,91.57,50.42,46.78,98.74,58.57



===================== MA10 Result =====================



Model,AdaBoost,Bagging,CatBoost,DecisionTree,LightGBM,RandomForest,SVM,XGBoost
Ticker,,,,,,,,
AAPL,-11.67,-1.32,-13.76,0.14,-10.48,-2.07,-14.04,-4.70
ADBE,-29.47,-10.72,-25.87,-11.00,-24.42,-9.19,-8.81,-24.54
AMD,17.50,15.39,-6.06,-1.73,11.09,19.09,7.14,-8.75
CRM,17.33,27.42,10.86,17.86,24.27,28.50,11.37,21.63
MSFT,17.45,18.27,11.36,17.96,18.50,17.10,17.63,19.92
NOW,29.10,20.47,1.04,8.11,-3.61,3.93,20.98,3.38
NVDA,-5.55,-3.25,-2.58,16.32,-1.94,-8.95,5.40,-11.02
ORCL,148.92,137.45,127.26,61.75,131.48,143.78,120.68,129.54



===================== MA20 Result =====================



Model,AdaBoost,Bagging,CatBoost,DecisionTree,LightGBM,RandomForest,SVM,XGBoost
Ticker,,,,,,,,
AAPL,0.27,2.55,3.38,10.60,-4.26,7.99,14.82,0.09
ADBE,-5.45,-7.68,-3.76,-8.33,-14.59,-7.18,-9.75,-5.37
AMD,5.56,17.49,15.44,26.12,20.59,16.33,5.59,9.69
CRM,3.00,-2.01,1.42,-9.91,0.67,1.60,7.61,-2.04
MSFT,-0.04,-0.57,-0.65,11.00,4.59,-1.86,0.51,-1.14
NOW,37.26,28.65,33.95,-5.77,23.82,32.39,27.60,32.45
NVDA,3.51,7.76,46.34,16.70,20.64,36.35,1.15,27.34
ORCL,61.28,66.28,67.45,66.99,55.33,71.94,79.99,67.60


In [16]:
# 1) Max Drawdown 
def max_drawdown(equity_curve):
    peak = equity_curve[0]
    max_dd = 0
    for value in equity_curve:
        if value > peak:
            peak = value
        drawdown = (peak - value) / peak
        max_dd = max(max_dd, drawdown)
    return max_dd * 100  # %

# 2) Backtest
def backtest_with_curve(prices, signals, initial_capital=100000):

    cash = initial_capital
    shares = 0
    position = 0
    equity_curve = []

    for i in range(len(signals)):
        price = prices[i]

        # Buy
        if signals[i] == 1 and position == 0:
            shares = cash / price
            cash = 0
            position = 1

        # Sell
        elif signals[i] == -1 and position == 1:
            cash = shares * price
            shares = 0
            position = 0

        # Track portfolio value daily
        total_value = cash + shares * price
        equity_curve.append(total_value)

    final_value = equity_curve[-1]
    return final_value, equity_curve

# 3) Result
results = []

for ticker in split_data.keys():

    df = labeled_data[ticker]
    test_len = len(split_data[ticker]["X_test"])
    close_prices = df["Close"].iloc[-test_len:].values

    for target in ["MA5_dir", "MA10_dir", "MA20_dir"]:
        for model_name, model_dict in all_models.items():

            model = model_dict[ticker][target]
            y_pred_cont = model.predict(split_data[ticker]["X_test"])
            signals = np.where(y_pred_cont > 0, 1, -1)

            final_val, curve = backtest_with_curve(close_prices, signals)
            ret = (final_val - 100000) / 100000 * 100

            # Year calculation
            years = test_len / 252
            annualized = ((final_val / 100000) ** (1 / years) - 1) * 100

            dd = max_drawdown(curve)

            results.append({
                "Model": model_name,
                "Return%": ret,
                "Annualized%": annualized,
                "MaxDD%": dd
            })

# 4) Table
df_final = pd.DataFrame(results)

summary = df_final.groupby("Model").agg({
    "Return%": "mean",
    "Annualized%": "mean",
    "MaxDD%": "mean"
}).reset_index()

summary["Balance"] = 100000 * (1 + summary["Return%"] / 100)

summary = summary[[
    "Model", "Balance", "Return%", "Annualized%", "MaxDD%"
]]

print("Average results for ML trading strategies without MacroEconomic indicator")

display(summary.style.format({
    "Balance": "{:,.2f}",
    "Return%": "{:.2f}",
    "Annualized%": "{:.2f}",
    "MaxDD%": "{:.2f}"
}))


Average results for ML trading strategies without MacroEconomic indicator


,Model,Balance,Return%,Annualized%,MaxDD%
0,AdaBoost,"116,913.21",16.91,14.44,26.14
1,Bagging,"117,627.24",17.63,15.15,25.08
2,CatBoost,"113,652.57",13.65,11.58,25.66
3,DecisionTree,"116,813.84",16.81,14.49,25.92
4,LightGBM,"112,953.95",12.95,11.04,26.51
5,RandomForest,"119,145.91",19.15,16.43,25.07
6,SVM,"117,771.93",17.77,15.21,25.86
7,XGBoost,"113,864.65",13.86,11.82,26.36
